# vocab.registry
> Typed, query-able vocabulary registry for CogitareLink.
---
Loads `data/registry_data.json` at import-time, validates each
entry with `pydantic`, and exposes convenient search helpers.

In [ ]:
#| default_exp vocab.registry

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations
import json, importlib.resources as pkg
from typing import List, Dict, Any
from pydantic import BaseModel, AnyUrl, Field, ValidationError
from cogitarelink.core.debug import get_logger
from urllib.parse import urlparse, urlunparse




In [ ]:
#| export
log = get_logger("registry")

# Load bundled JSON once
_RAW_JSON: Dict[str, Any] = json.loads(
    pkg.files("cogitarelink").joinpath("data/registry_data.json").read_text()
)

## VocabEntry

In [ ]:
#| export
class VocabEntry(BaseModel):
    uri: AnyUrl
    alternative_uris: List[AnyUrl] = []
    prefix: str
    title: str
    description: str
    version: str
    publisher: str
    support_level: str            # "direct" | "cache" | "github_raw" etc.
    resources: Dict[str, Any]
    access_patterns: Dict[str, Any]
    url_transformations: List[Dict[str, str]] = []
    features: Dict[str, bool]
    common_terms: List[str] = Field(default_factory=list)
    common_types: List[str] = Field(default_factory=list)
    related_vocabs: List[str] = Field(default_factory=list)

In [ ]:
#| export
def _norm(u:str) -> str:
    """Lower-case scheme+host and drop a single trailing '/'."""
    p = urlparse(u)
    scheme, netloc, path = p.scheme.lower(), p.netloc.lower(), p.path
    if path.endswith("/") and path != "/":
        path = path[:-1]
    return urlunparse((scheme, netloc, path, "", "", ""))


In [ ]:
#| export
class VRegistry:
    "Immutable mapping of `prefix -> VocabEntry`."
    def __init__(self, raw: Dict[str, Any] | None = None):
        raw = raw or _RAW_JSON
        self._data: Dict[str, VocabEntry] = {}
        invalid: Dict[str, str] = {}
        for k, v in raw.items():
            try:
                self._data[k] = VocabEntry(**v)
            except ValidationError as e:
                invalid[k] = e.json()
        if invalid:
            log.warning("Skipped %d invalid vocab entries", len(invalid))
        # build once after all valid entries were loaded
        def _to_str_list(entry):
            "Return main + alt URIs as plain strings"
            return [str(entry.uri), *[str(u) for u in entry.alternative_uris]]

        self._alt_uri_map = {
            _norm(url): entry
            for entry in self._data.values()
            for url   in _to_str_list(entry)
        }
        
    # ------------- basic look-ups ----------------------------------
    def by_prefix(self, p: str) -> VocabEntry:
        return self._data[p]
    
    def by_uri(self, uri:str) -> VocabEntry:
        return self._alt_uri_map[_norm(uri)]
    
    def search(self, kw: str) -> List[VocabEntry]:
        kw = kw.lower()
        return [v for v in self._data.values()
                if kw in v.description.lower() or kw in v.title.lower()]

    # ------------- dunder helpers ----------------------------------
    def __iter__(self): return iter(self._data)
    def __getitem__(self, k): return self._data[k]
    def __len__(self): return len(self._data)

# global singleton
registry = VRegistry()

In [ ]:
#| hide
from fastcore.test import test, test_eq, test_ne, test_is, test_fail
from operator import eq
from cogitarelink.vocab.registry import registry, VocabEntry
test_is(isinstance(registry["schema"], VocabEntry), True)
test(len(registry)>5, True, eq, "registry should have >5 vocabs")
schema=registry.by_prefix("schema")
test_eq(schema.prefix,"schema")
test_eq(schema.resources["context"].split("/")[2],"schema.org")
alt=registry.by_uri("http://schema.org/")
test_is(alt,schema)
hits=registry.search("supply chain")
test_ne(len(hits),0)
test_eq(any(v.prefix=="epcis" for v in hits),True)
test_fail(lambda:registry.by_prefix("no-such-vocab"),contains="KeyError")

KeyError: 'http://schema.org/'

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()